In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import mne
import torch
from pathlib import Path
from dn3_ext import BENDRClassification

data_dir = Path("data")
edf_path_demo = data_dir / "test_from_20s.edf"
raw_demo = mne.io.read_raw_edf(edf_path_demo, preload=True, verbose=False)
raw_demo.drop_channels(['ECG  ECG', 'EEG OZ-A1']) # TODO convert to meaningfull channels
weights_dir = Path("pretrained_weights")

In [ ]:
arr: np.ndarray = raw_demo.get_data()  # shape: channels x samples
x = torch.from_numpy(arr[np.newaxis, ...]).float()  # shape: batch x channels x samples

model = BENDRClassification(targets=2, samples=arr.shape[1], channels=arr.shape[0])
# model.load_pretrained_modules(weights_dir / "encoder.pt",
#                               weights_dir / "contextualizer.pt",
#                               strict=False)

device = "cpu" # TODO if torch.cuda.is_available, etc

enc_sd = torch.load(str(weights_dir / "encoder.pt"), map_location="cpu")
model.encoder.load_state_dict(enc_sd, strict=False)

ctx_sd = torch.load(str(weights_dir / "contextualizer.pt"), map_location="cpu")
model.contextualizer.load_state_dict(ctx_sd, strict=False)

# make sure `mask_replacement` is not trainable
if hasattr(model.contextualizer, "mask_replacement"):
    model.contextualizer.mask_replacement.requires_grad = False

model.eval()
with torch.no_grad():
    feats = model.features_forward(x)

print("feature shape:", feats.shape)
print("first values:", feats.flatten()[:10])